In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
eeg_eye_state = fetch_ucirepo(id=264) 
  
# data (as pandas dataframes) 
X = eeg_eye_state.data.features 
y = eeg_eye_state.data.targets 
  
# metadata 
print(eeg_eye_state.metadata) 
  
# variable information 
print(eeg_eye_state.variables) 

{'uci_id': 264, 'name': 'EEG Eye State', 'repository_url': 'https://archive.ics.uci.edu/dataset/264/eeg+eye+state', 'data_url': 'https://archive.ics.uci.edu/static/public/264/data.csv', 'abstract': 'The data set consists of 14 EEG values and a value indicating the eye state.', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 14980, 'num_features': 14, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['eyeDetection'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2013, 'last_updated': 'Thu Mar 21 2024', 'dataset_doi': '10.24432/C57G7J', 'creators': ['Oliver Roesler'], 'intro_paper': None, 'additional_info': {'summary': "All data is from one continuous EEG measurement with the Emotiv EEG Neuroheadset. The duration of the measurement was 117 seconds. The eye state was detected via a camera during the EEG measuremen

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [3]:
features = X
feature_names = X.columns

df = pd.DataFrame(features,columns = feature_names)
df.target = y
df.head()

/var/folders/4y/0qtvj2hs6r9_dvmt7n7fbdk40000gn/T/ipykernel_25068/2312714648.py:5: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.target = y


,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,4329.23,4009.23,4289.23,4148.21,4350.26,4586.15,4096.92,4641.03,4222.05,4238.46,4211.28,4280.51,4635.90,4393.85
1,4324.62,4004.62,4293.85,4148.72,4342.05,4586.67,4097.44,4638.97,4210.77,4226.67,4207.69,4279.49,4632.82,4384.10
2,4327.69,4006.67,4295.38,4156.41,4336.92,4583.59,4096.92,4630.26,4207.69,4222.05,4206.67,4282.05,4628.72,4389.23
3,4328.72,4011.79,4296.41,4155.90,4343.59,4582.56,4097.44,4630.77,4217.44,4235.38,4210.77,4287.69,4632.31,4396.41
4,4326.15,4011.79,4292.31,4151.28,4347.69,4586.67,4095.90,4627.69,4210.77,4244.10,4212.82,4288.21,4632.82,4398.46


In [4]:
fs = 128

In [5]:
beta_data = []

for i in X.columns:
    signal = X[i].values
    N = len(signal)

    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(N, d=1/fs)

    beta_mask = (freqs >= 13) & (freqs <= 30)

    beta_fft = np.zeros_like(fft_vals)
    beta_fft[beta_mask] = fft_vals[beta_mask]

    beta_signal = np.fft.ifft(beta_fft).real

    beta_data.append(beta_signal)

In [6]:
beta_data = np.array(beta_data)
beta_data = np.transpose(beta_data)

features = beta_data
feature_names = X.columns

df = pd.DataFrame(features,columns = feature_names)
df.head(10)

,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,15.103843,1.045047,3.101060,4.191812,1.441866,-40.990501,3.920292,2.047254,12.999068,1.806480,4.064590,1.927087,10.228758,-74.528410
1,4.956922,0.821675,3.562179,4.041567,0.204921,17.159152,2.188774,0.253273,-0.581134,-1.587264,4.520225,-0.172720,4.068479,45.296543
2,7.846711,-0.008771,1.874768,34.298267,-1.225110,-16.117878,28.412447,-2.149483,1.312993,-3.698994,1.482535,-1.505892,3.873331,-18.484753
3,22.493088,0.048855,0.085062,38.935411,-0.875537,-27.267144,33.651823,-2.864637,16.263720,-1.457118,-0.336517,0.342528,10.837861,-45.561686
4,15.510521,0.399671,-1.477248,-6.335958,0.571069,61.256809,-4.971887,-1.117111,13.368910,2.334461,0.079951,2.423471,7.482559,117.371471
5,-14.633830,-0.705749,-3.052028,-43.503368,1.012222,127.144922,-37.703494,1.078945,-9.424266,2.737270,-0.512016,0.517532,-8.477525,238.282185
6,-28.792295,-2.488684,-3.413676,-27.624567,0.063613,53.250186,-24.130084,1.590063,-19.806859,-0.132183,-3.217552,-3.662926,-16.851993,93.494329
7,-11.771974,-1.809558,-1.138492,0.577125,-0.644174,-61.372660,1.303160,1.001255,-6.741636,-1.521385,-4.184519,-3.847177,-7.825524,-125.633719
8,4.049521,1.869148,2.626476,-6.982599,-0.114847,-65.842121,-4.593077,1.134895,2.869455,0.480410,-0.374824,1.477991,2.997726,-129.256382
9,-2.773090,4.387324,4.511218,-19.854419,0.617785,-7.493599,-17.059055,1.659362,-5.346794,2.162426,4.749941,5.680399,2.354200,-11.215042


In [7]:
y = y['eyeDetection']
beta_data_train, beta_data_test, y_train, y_test = train_test_split(beta_data, y, test_size=0.2, random_state=67)

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(beta_data_train)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [9]:
beta_data_train = scaler.transform(beta_data_train)
beta_data_test = scaler.transform(beta_data_test)

from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [10]:
xgb.fit(beta_data_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [11]:
y_pred = xgb.predict(beta_data_test)
from sklearn import metrics
metrics.accuracy_score(y_test, y_pred)

0.6665554072096128

In [12]:
nobeta_data = []

for i in X.columns:
    signal = X[i].values
    N = len(signal)

    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(N, d=1/fs)

    nobeta_mask = (freqs >= 30) | (freqs <= 13)

    nobeta_fft = np.zeros_like(fft_vals)
    nobeta_fft[nobeta_mask] = fft_vals[nobeta_mask]

    nobeta_signal = np.fft.ifft(nobeta_fft).real

    nobeta_data.append(nobeta_signal)

In [13]:
nobeta_data = np.array(nobeta_data)
nobeta_data = np.transpose(nobeta_data)

features = nobeta_data
feature_names = X.columns

df = pd.DataFrame(features,columns = feature_names)
df.head(10)

,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,4314.126157,4008.184953,4286.128940,4144.018188,4348.818134,4627.140501,4092.999708,4638.982746,4209.050932,4236.653520,4207.215410,4278.582913,4625.671242,4468.378410
1,4319.663078,4003.798325,4290.287821,4144.678433,4341.845079,4569.510848,4095.251226,4638.716727,4211.351134,4228.257264,4203.169775,4279.662720,4628.751521,4338.803457
2,4319.843289,4006.678771,4293.505232,4122.111733,4338.145110,4599.707878,4068.507553,4632.409483,4206.377007,4225.748994,4205.187465,4283.555892,4624.846669,4407.714753
3,4306.226912,4011.741145,4296.324938,4116.964589,4344.465537,4609.827144,4063.788177,4633.634637,4201.176280,4236.837118,4211.106517,4287.347472,4621.472139,4441.971686
4,4310.639479,4011.390329,4293.787248,4157.615958,4347.118931,4525.413191,4100.871887,4628.807111,4197.401090,4241.765539,4212.740049,4285.786529,4625.337441,4281.088529
5,4335.663830,4005.325749,4287.152028,4196.833368,4344.627778,4460.035078,4131.033494,4615.841055,4211.984266,4230.082730,4210.252016,4280.512468,4636.687525,4151.457815
6,4348.282295,4003.518684,4283.923676,4179.414567,4343.526387,4531.369814,4113.870084,4614.309937,4232.116859,4226.802183,4204.247552,4273.402926,4641.981993,4284.965671
7,4337.411974,4008.479558,4279.598492,4142.502875,4344.744174,4644.452660,4085.876840,4613.868745,4212.381636,4231.781385,4200.084519,4270.517177,4629.875524,4506.143719
8,4322.100479,4008.900852,4273.783524,4146.472599,4345.244847,4649.942121,4095.873077,4607.075105,4184.820545,4229.259590,4202.424824,4272.372009,4624.182274,4518.996382
9,4328.923090,4006.892676,4272.408782,4161.904419,4343.482215,4590.053599,4109.879055,4607.060638,4199.706794,4226.557574,4208.070059,4272.269601,4635.085800,4404.545042


In [14]:
nobeta_data_train, nobeta_data_test, y_train, y_test = train_test_split(nobeta_data, y, test_size=0.2, random_state=67)

In [15]:
scaler = StandardScaler()
scaler.fit(nobeta_data_train)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [16]:
nobeta_data_train = scaler.transform(nobeta_data_train)
nobeta_data_test = scaler.transform(nobeta_data_test)

xgb = XGBClassifier()
xgb

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [17]:
xgb.fit(nobeta_data_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [18]:
y_pred = xgb.predict(nobeta_data_test)
metrics.accuracy_score(y_test, y_pred)

0.8911882510013351

In [20]:
import random, os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn import metrics

def seed_everything(seed_value):
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)

seeds = [67, 69, 41, 523, 0]

results = []

for seed in seeds:
    seed_everything(seed)

    b_train, b_test, y_tr, y_te = train_test_split(beta_data, y, test_size=0.2, random_state=seed)
    sc_b = StandardScaler().fit(b_train)
    b_train_sc = sc_b.transform(b_train)
    b_test_sc  = sc_b.transform(b_test)

    clf_beta = XGBClassifier(random_state=seed)
    clf_beta.fit(b_train_sc, y_tr)
    acc_beta = metrics.accuracy_score(y_te, clf_beta.predict(b_test_sc))

    nb_train, nb_test, y_tr2, y_te2 = train_test_split(nobeta_data, y, test_size=0.2, random_state=seed)
    sc_nb = StandardScaler().fit(nb_train)
    nb_train_sc = sc_nb.transform(nb_train)
    nb_test_sc  = sc_nb.transform(nb_test)

    clf_nobeta = XGBClassifier(random_state=seed)
    clf_nobeta.fit(nb_train_sc, y_tr2)
    acc_nobeta = metrics.accuracy_score(y_te2, clf_nobeta.predict(nb_test_sc))

    results.append((seed, acc_beta, acc_nobeta))
    print(f"Seed {seed} | Beta: {acc_beta} | NoBeta: {acc_nobeta}")

Seed 67 | Beta: 0.6665554072096128 | NoBeta: 0.8911882510013351
Seed 69 | Beta: 0.6722296395193591 | NoBeta: 0.8928571428571429
Seed 41 | Beta: 0.6759012016021362 | NoBeta: 0.8881842456608812
Seed 523 | Beta: 0.6698931909212283 | NoBeta: 0.9025367156208278
Seed 0 | Beta: 0.670894526034713 | NoBeta: 0.8848464619492656
